# EDA PRECIOS HISTÓRICOS 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

### Carga y exploración de estructura

In [2]:
df = pd.read_parquet("../data/processed/01_consolidated/dataset_precios.parquet")

print(f"\n ESTRUCTURA PRECIOS")
print(f"  Filas: {len(df):,}")
print(f"  Columnas: {df.shape[1]}")
print(f"  Tamaño memoria: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
df.head()



 ESTRUCTURA PRECIOS
  Filas: 294
  Columnas: 8
  Tamaño memoria: 0.03 MB


,distrito,año,precio_m2_eur,fuente,fecha_actualizacion,precio_anterior,cambio_absoluto,pct_cambio_yoy
0,Arganzuela,2013,2850,Colegio de Registradores / TINSA,2026-08-27,NaN,NaN,NaN
1,Arganzuela,2014,2980,Colegio de Registradores / TINSA,2026-08-27,2850.0,130.0,4.56
2,Arganzuela,2015,3150,Colegio de Registradores / TINSA,2026-08-27,2980.0,170.0,5.70
3,Arganzuela,2016,3450,Colegio de Registradores / TINSA,2026-08-27,3150.0,300.0,9.52
4,Arganzuela,2017,3780,Colegio de Registradores / TINSA,2026-08-27,3450.0,330.0,9.57


### Análisis de nulos

In [3]:

nulos = df.isnull().sum()   
if nulos.sum() == 0:
    print(f"\n Sin valores nulos")
    
print(f"\n  Valores nulos por columna:")
nulos_df = pd.DataFrame({
        'Columna': nulos.index,
        'Nulos': nulos.values,
        '%': (nulos.values / len(df) * 100).round(2)
    })
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Nulos', ascending=False)
    
for _, row in nulos_df.iterrows():
        print(f"  {row['Columna']:30s} {row['Nulos']:>8,d}  ({row['%']:>6.2f}%)")


  Valores nulos por columna:
  precio_anterior                      21  (  7.14%)
  cambio_absoluto                      21  (  7.14%)
  pct_cambio_yoy                       21  (  7.14%)


### Análisis de duplicados

In [4]:
duplicados_totales = df.duplicated().sum()
print(f"  Total (todas las columnas): {duplicados_totales:,}")

  Total (todas las columnas): 0


### Análisis de tipos de datos

In [5]:
print(f"\n Tipos de datos:")
tipos = df.dtypes.value_counts()
for tipo, cantidad in tipos.items():
    print(f"  {str(tipo):25s} {cantidad:3d} columnas")

df.info()


 Tipos de datos:
  str                         3 columnas
  float64                     3 columnas
  int64                       2 columnas
<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   distrito             294 non-null    str    
 1   año                  294 non-null    int64  
 2   precio_m2_eur        294 non-null    int64  
 3   fuente               294 non-null    str    
 4   fecha_actualizacion  294 non-null    str    
 5   precio_anterior      273 non-null    float64
 6   cambio_absoluto      273 non-null    float64
 7   pct_cambio_yoy       273 non-null    float64
dtypes: float64(3), int64(2), str(3)
memory usage: 33.3 KB


### Análisis de cobertura geografica

In [6]:
print(f"\n  COBERTURA GEOGRÁFICA:")
if 'distrito' in df.columns:
        distritos = df['distrito'].nunique()
        print(f"  Distritos: {distritos}")
        
        print(f"\n  Distritos únicos:")
        for dist in sorted(df['distrito'].unique()):
            count = len(df[df['distrito'] == dist])
            print(f"    {dist:30s} {count:>3} registros")   



  COBERTURA GEOGRÁFICA:
  Distritos: 21

  Distritos únicos:
    Arganzuela                      14 registros
    Barajas                         14 registros
    Carabanchel                     14 registros
    Centro                          14 registros
    Chamartín                       14 registros
    Chamberí                        14 registros
    Ciudad Lineal                   14 registros
    Fuencarral                      14 registros
    Hortaleza                       14 registros
    Latina                          14 registros
    Moncloa                         14 registros
    Moratalaz                       14 registros
    Puente de Vallecas              14 registros
    Retiro                          14 registros
    Salamanca                       14 registros
    San Blas                        14 registros
    Tetuán                          14 registros
    Usera                           14 registros
    Vicálvaro                       14 registros
    Vil

### Análisis cobertura temporal

In [7]:
print(f"\n COBERTURA TEMPORAL:")
    
if 'año' in df.columns:
    anios = sorted(df['año'].unique())
    print(f"  Años: {anios[0]} → {anios[-1]} ({len(anios)} años)")
        
    # Verificar continuidad
    anios_esperados = set(range(anios[0], anios[-1] + 1))
    anios_reales = set(anios)
    faltantes = anios_esperados - anios_reales
    if faltantes:
        print(f" Años faltantes: {sorted(faltantes)}")
    else:
        print(f" Cobertura completa sin gaps")



 COBERTURA TEMPORAL:
  Años: 2013 → 2026 (14 años)
 Cobertura completa sin gaps


### Análisis de precios (€/m²)

In [8]:
print(f"\n ÁREA POR DISTRITO:")
print(f"\n💶 ANÁLISIS DE PRECIOS (€/m²):")
    
if 'precio_m2_eur' in df.columns:
    precios = df['precio_m2_eur'].dropna()
        
    print(f"  Min: {precios.min():>10,.0f} €/m²")
    print(f"  Max: {precios.max():>10,.0f} €/m²")
    print(f"  Media: {precios.mean():>9,.0f} €/m²")
    print(f"  Mediana: {precios.median():>7,.0f} €/m²")
    print(f"  Desvío: {precios.std():>9,.0f} €/m²")
        
    # Por distrito por año
    if 'distrito' in df.columns and 'año' in df.columns:
        df_2026 = df[df['año'] == 2026]
        if len(df_2026) > 0:
            print(f"\n  Precios 2026 por distrito:")
            precios_2026 = df_2026.sort_values('precio_m2_eur', ascending=False)
            for _, row in precios_2026.iterrows():
                print(f"    {row['distrito']:30s} {row['precio_m2_eur']:>10,.0f} €/m²")


 ÁREA POR DISTRITO:

💶 ANÁLISIS DE PRECIOS (€/m²):
  Min:      1,550 €/m²
  Max:      8,320 €/m²
  Media:     4,378 €/m²
  Mediana:   4,380 €/m²
  Desvío:     1,560 €/m²

  Precios 2026 por distrito:
    Centro                              8,320 €/m²
    Chamberí                            7,950 €/m²
    Retiro                              7,610 €/m²
    Chamartín                           7,520 €/m²
    Salamanca                           6,769 €/m²
    Tetuán                              6,350 €/m²
    Arganzuela                          6,250 €/m²
    Moncloa                             6,050 €/m²
    Carabanchel                         5,850 €/m²
    Hortaleza                           5,850 €/m²
    Latina                              5,750 €/m²
    Ciudad Lineal                       5,750 €/m²
    Fuencarral                          5,350 €/m²
    Moratalaz                           5,250 €/m²
    San Blas                            5,150 €/m²
    Barajas                       

### Análisis de cambios de año

In [9]:
print(f"\n CAMBIOS AÑO A AÑO:")
    
if 'pct_cambio_yoy' in df.columns:
    cambios = df['pct_cambio_yoy'].dropna()
        
    if len(cambios) > 0:
        print(f"  Min: {cambios.min():>10.2f}%")
        print(f"  Max: {cambios.max():>10.2f}%")
        print(f"  Media: {cambios.mean():>9.2f}%")
        print(f"  Mediana: {cambios.median():>7.2f}%")
            
        # Años con mayor/menor crecimiento
        positivos = (cambios > 0).sum()
        negativos = (cambios < 0).sum()
        print(f"\n  Cambios positivos: {positivos}")
        print(f"  Cambios negativos: {negativos}")



 CAMBIOS AÑO A AÑO:
  Min:       0.28%
  Max:      15.09%
  Media:      6.82%
  Mediana:    6.67%

  Cambios positivos: 273
  Cambios negativos: 0


### Análisis de colatilidad por distrito

In [10]:
print(f"\n VOLATILIDAD POR DISTRITO:")
    
if 'distrito' in df.columns and 'precio_m2_eur' in df.columns:
    volatilidad = df.groupby('distrito')['precio_m2_eur'].std()
    volatilidad = volatilidad.sort_values(ascending=False)
        
    print(f"  Mayor volatilidad (desviación €/m²):")
    for dist, vol in volatilidad.head(5).items():
        print(f"    {dist:30s} ±{vol:>8,.0f} €/m²")


 VOLATILIDAD POR DISTRITO:
  Mayor volatilidad (desviación €/m²):
    Chamberí                       ±   1,382 €/m²
    Salamanca                      ±   1,341 €/m²
    Chamartín                      ±   1,325 €/m²
    Arganzuela                     ±   1,244 €/m²
    Tetuán                         ±   1,244 €/m²
